In [0]:
# Read all CSV files from Catalog
flights_df = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv('/Volumes/flight-delay-analytics-catalog/bronze/raw_files/flights.csv')
)

airlines_df = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv('/Volumes/flight-delay-analytics-catalog/bronze/raw_files/airlines.csv')
)

airports_df = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv('/Volumes/flight-delay-analytics-catalog/bronze/raw_files/airports.csv')
)

cancellation_codes_df = (
    spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv('/Volumes/flight-delay-analytics-catalog/bronze/raw_files/cancellation_codes.csv')
)

In [0]:
# Check schema of all spark tables created
flights_df.printSchema()
airlines_df.printSchema()
airports_df.printSchema()
cancellation_codes_df.printSchema()

In [0]:
# Check shape of each spark df
dfs = {
    "flights": flights_df,
    "airlines": airlines_df,
    "airports": airports_df,
    "cancellation_codes": cancellation_codes_df
}

for name, df in dfs.items():
    print(f"{name}: \t\tRows: {df.count():,}, \t\tCols: {len(df.columns)}")

In [0]:

# Check shape of each dataframe
dfs = {
    "flights": flights_df,
    "airlines": airlines_df,
    "airports": airports_df,
    "cancellation_codes": cancellation_codes_df
}

shape = [(name, df.count(), len(df.columns)) for name, df in dfs.items()]
display(spark.createDataFrame(shape, ['name', 'rows', 'cols']))

**Basic Data Profiling**

In [0]:
# Check and inspect for missing values
# null values on aiports_df
from pyspark.sql import functions as F

display(airports_df.select(
    [F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}') 
    for column in airports_df.columns]
))

In [0]:
# Inspect rows with null values on airports_df

display(airports_df.filter(
    F.col('LATITUDE').isNull() |
    F.col('LONGITUDE').isNull()
))

In [0]:
# Check for null values manually on small tables airlines_df and cancellation_codes_df
display(airlines_df)
display(cancellation_codes_df)

In [0]:
# Check null values on flights_df
display(flights_df.select(
    [F.sum(F.col(column).isNull().cast('int')).alias(f'missing_{column}')
    for column in flights_df.columns]
))

In [0]:
display(flights_df.limit(10))

In [0]:
# inspect missing values on flights_df
# check if all flights that did not get cancelled does not have a cancellation reason
display(flights_df.filter(
    (F.col('CANCELLED') == 0) &
    (~F.col('CANCELLATION_REASON').isNull())
))

In [0]:
# check if all flights that got cancelled have a cancellation reason
display(flights_df.filter(
    (F.col('CANCELLED') == 1) &
    (F.col('CANCELLATION_REASON').isNull())
))

In [0]:
# Check for duplicates

for name, df in dfs.items():
    print(f'{name}: Duplicates = {df.count() - df.dropDuplicates().count()}')

In [0]:
from pyspark.sql import functions as F

In [0]:
# Check range of columns with known values

year_range = flights_df.select(
    F.lit('Year').alias('Metric'),
    F.min('YEAR').alias('Min'),
    F.max('YEAR').alias('Max')
)

month_range = flights_df.select(
    F.lit('Month').alias('Metric'),
    F.min('MONTH').alias('Min'),
    F.max('MONTH').alias('Max')
)

day_range = flights_df.select(
    F.lit('Day').alias('Metric'),
    F.min('DAY').alias('Min'),
    F.max('DAY').alias('Max')
)

day_of_week_range = flights_df.select(
    F.lit('Day of Week').alias('Metric'),
    F.min('DAY_OF_WEEK').alias('Min'),
    F.max('DAY_OF_WEEK').alias('Max')
)

metrics = (
    year_range
    .union(month_range)
    .union(day_range)
    .union(day_of_week_range)
)

display(metrics)

In [0]:
# Check for referential integrity on flights_df airlines column
airlines = [
    row["IATA_CODE"]
    for row in airlines_df.select("IATA_CODE").collect()
]

display(
    flights_df.filter(
        ~F.col("AIRLINE").isin(airlines)
    )
)

In [0]:
# Check for referential integrity on airports column
airports = [
    row['IATA_CODE']
    for row in airports_df.select('IATA_CODE').collect()
]

display(flights_df.filter(
    ~F.col('ORIGIN_AIRPORT').isin(airports)
))

NOTE: failed to match the standard 3-letter IATA codes because the data collection format switched to 5-digit US DOT Bureau of Transportation Statistics (BTS) airport IDs starting in October 2015

In [0]:
# Double check if airport codes changed format mid-year
sep_flights = flights_df.filter(
    F.col('MONTH') == 10
).limit(5)

oct_flights = flights_df.filter(
    F.col('MONTH') == 9
).limit(5)

display(
    sep_flights.union(oct_flights)
)

In [0]:
# Save into tables
(flights_df.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("`flight-delay-analytics-catalog`.bronze.flights")
)

(airlines_df.write
.format('delta')
.mode('overwrite')
.saveAsTable("`flight-delay-analytics-catalog`.bronze.airlines"))

(airports_df.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("`flight-delay-analytics-catalog`.bronze.airports")
)

(cancellation_codes_df.write
.format('delta')
.mode('overwrite')
.saveAsTable("`flight-delay-analytics-catalog`.bronze.cancellation_codes")
)


In [0]:
%sql
-- verify existence of tables
SHOW TABLES IN `flight-delay-analytics-catalog`.bronze;

Check row count of each table

In [0]:
%sql
-- check row count of each table if it matches shape of spark dfs
SELECT 'airlines' as table_name, COUNT(*) as row_count
FROM `flight-delay-analytics-catalog`.bronze.airlines

UNION ALL

SELECT 'airports' as table_name, COUNT(*) as row_count
FROM `flight-delay-analytics-catalog`.bronze.airports

UNION ALL

SELECT 'cancellation_codes' as table_name, COUNT(*) as row_count
FROM `flight-delay-analytics-catalog`.bronze.cancellation_codes

UNION ALL

SELECT 'flights' as table_name, COUNT(*) as row_count
FROM `flight-delay-analytics-catalog`.bronze.flights

In [0]:
%sql
-- double check schema of airports table
DESCRIBE `flight-delay-analytics-catalog`.bronze.airports;

In [0]:
%sql
-- double check schema of airlines table
DESCRIBE `flight-delay-analytics-catalog`.bronze.airlines;

In [0]:
%sql
-- double check schema of flights table
DESCRIBE `flight-delay-analytics-catalog`.bronze.flights;

In [0]:
%sql
-- double check schema of cancellation_codes table
DESCRIBE `flight-delay-analytics-catalog`.bronze.cancellation_codes;